In [0]:
import pandas as pd
import numpy as np

In [0]:
# Uploading tables

customers = spark.table("store_performance.store_perfomance_casestudy.customers_dataset").toPandas()
orders = spark.table("store_performance.store_perfomance_casestudy.orders_dataset").toPandas()
payments = spark.table("store_performance.store_perfomance_casestudy.payments_dataset").toPandas()
products = spark.table("store_performance.store_perfomance_casestudy.products_dataset").toPandas()

In [0]:
customers.head()


In [0]:
orders.head()

In [0]:
payments.head()

In [0]:
products.head()

In [0]:
print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Payments:", payments.shape)
print("Products:", products.shape)

In [0]:

## Looking at the columns
print(customers.columns)
print(orders.columns)
print(payments.columns)
print(products.columns)

In [0]:
# Checking data types for my customers dataset

customers.info()

In [0]:
# Checking data types for my orders dataset


orders.info()

In [0]:
## Checking data types for my Payments dataset

payments.info()

In [0]:
## Checking data types for my Products dataset

products.info()

1. Checking for Missing Values

In [0]:
# missing values on the customers dataset.

customers.isna().sum()

In [0]:
# missing values on the Orders dataset.

orders.isna().sum()

In [0]:
# Checking for missing values on the payments dataset
payments.isna().sum()

In [0]:
#Checking for missing values on the products dataset
products.isna().sum()

In [0]:
# Checking for duplicates on the orders dataset

orders.duplicated().sum()

In [0]:
orders[orders.duplicated(keep=False)].sort_values("OrderID")

orders_clean = orders.drop_duplicates()
print(orders.shape)
print(orders_clean.shape)

In [0]:
orders_clean["OrderID"].duplicated().sum()

In [0]:
orders_clean["Quantity"].describe()

In [0]:
orders_clean[orders_clean["Quantity"] <= 0]

In [0]:
###Checking for Discount

orders_clean["Discount"].describe()

In [0]:
# Checking for product prices

products["UnitPrice"].describe()

In [0]:
#Checking for Customer ages

customers["Age"].describe()

In [0]:
#Checking for incosistent cities

customers["City"].value_counts(dropna=False)

In [0]:
### Standardising the cities.

customers_clean = customers.copy()

customers_clean["City"] = customers_clean["City"].replace({
    "tehran": "Tehran",
    "Mashad": "Mashhad"
})

customers_clean["City"].value_counts(dropna=False)

In [0]:
### Checking for Customer Links

orders_clean[
    ~orders_clean["CustomerID"].isin(customers_clean["CustomerID"])
]


In [0]:
# Checking for broken productIDs
orders_clean[
    ~orders_clean["ProductID"].isin(products["ProductID"])
]


In [0]:
## Checking payment links

payments[
    ~payments["OrderID"].isin(orders_clean["OrderID"])
]

3. Join 1 - orders & products

In [0]:
## Because we want to keep our orders and bring product information into them.
shop = orders_clean.merge(
    products,
    on="ProductID",
    how="left"
)
print(shop.shape)

4. Joining Customers table

In [0]:
# Joining the customers table to the joined order & Products (shop)
shop = shop.merge(
    customers_clean[[
        "CustomerID",
        "City",
        "CustomerSegment"
    ]],
    on="CustomerID",
    how="left"
)
print(shop.shape)

5.Joining the Payments table on the (customer, order& products table joined)

In [0]:
shop = shop.merge(
    payments[[
        "OrderID",
        "PaymentStatus"
    ]],
    on="OrderID",
    how="left"
)
print(shop.shape)

4. Creating Calculated values / columns

In [0]:

##Revenue  

shop["Revenue"] = (
    shop["Quantity"]
    * shop["UnitPrice"]
    * (1 - shop["Discount"])
)

In [0]:
##Extracting the year and  date functions
shop["OrderDate"] = pd.to_datetime(
    shop["OrderDate"],
    errors="coerce"
)

shop["Year"] = shop["OrderDate"].dt.year

In [0]:
# extracting the month from the order date column
shop["Month"] = shop["OrderDate"].dt.to_period("M")

In [0]:
#Completed
   #   +
#Paid
    #  +
#Valid Quantity
     # +
#Valid Discount

# defines my revenue
shop["valid_quantity"] = (
    shop["Quantity"].notna()
    & (shop["Quantity"] > 0)
)

shop["valid_discount"] = (
    shop["Discount"].notna()
    & shop["Discount"].between(0, 1)
)

In [0]:
shop["revenue_eligible"] = (
    (shop["Status"] == "Completed")
    & (shop["PaymentStatus"] == "Paid")
    & shop["valid_quantity"]
    & shop["valid_discount"]
)

In [0]:
#### Calculating Revenue

shop["Revenue"] = (
    shop["Quantity"]
    * shop["UnitPrice"]
    * (1 - shop["Discount"])
)


In [0]:
#Creating our revenue dataset

revenue_data = shop[
    shop["revenue_eligible"]
].copy()

print(revenue_data.shape)

In [0]:
## Total Revenue

total_revenue = revenue_data["Revenue"].sum()

print(total_revenue)

In [0]:
number_of_orders = revenue_data["OrderID"].nunique()

print(number_of_orders)

In [0]:
## Average order value = Total revnue / ## of Orders

aov =round ((total_revenue / number_of_orders),1)

print(aov)

In [0]:
## Creating monthly revenue (montlhy column)

revenue_data["YearMonth"] = ( revenue_data["OrderDate"].dt.to_period("M"))


monthly_revenue = (
    revenue_data
    .dropna(subset=["OrderDate"])
    .groupby("YearMonth")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("OrderID", "nunique"),
        Units=("Quantity", "sum")
    )
    .reset_index()
)

In [0]:
## Calculating Montlhy AOV

monthly_revenue["AOV"] = (
    monthly_revenue["Revenue"]
    / monthly_revenue["Orders"]
)

monthly_revenue.head()

In [0]:
# find the highest-revenue month

monthly_revenue.loc[
    monthly_revenue["Revenue"].idxmax()
]

In [0]:
#find the lowest-revenue month

monthly_revenue.loc[
    monthly_revenue["Revenue"].idxmin()
]

In [0]:
# Comparing the first 6 months 

monthly_revenue["Year"] = (
    monthly_revenue["YearMonth"].dt.year
)

monthly_revenue["Month"] = (
    monthly_revenue["YearMonth"].dt.month
)

h1 = (
    monthly_revenue[
        monthly_revenue["Month"] <= 6
    ]
    .groupby("Year")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("Orders", "sum")
    )
    .reset_index()
)
display(h1)

In [0]:
#Revenue change in percentage

revenue_change = (
    (462194.20 - 624066.55)
    / 624066.55
) * 100

print(revenue_change)


6. Line Charts trends over time

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(
    monthly_revenue["YearMonth"].astype(str),
    monthly_revenue["Revenue"]
)

plt.title("Monthly Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue (R)")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

5. Products and Catergories

In [0]:
product_performance = (
    revenue_data
    .groupby(
        ["ProductID", "ProductName", "Category"]
    )
    .agg(
        Revenue=("Revenue", "sum"),
        Units=("Quantity", "sum"),
        Orders=("OrderID", "nunique")
    )
    .reset_index()
)
product_performance.head()

In [0]:
#Sorting by Revenue Largest to Lowest

top_products_revenue = (
    product_performance
    .sort_values(
        "Revenue",
        ascending=False
    )
)

top_products_revenue.head(10)

In [0]:
# Displaying  the top 5 performing products in revenue

top_5_revenue = (
    product_performance
    .sort_values(
        "Revenue",
        ascending=False
    )
    .head(5)
)

display(top_5_revenue)

In [0]:
# Units Sold largest to smallest

top_products_units = (
    product_performance
    .sort_values(
        "Units",
        ascending=False
    )
)
display(top_products_units.head(10))

In [0]:
# Category performance group by products & sort by revenue to determine how the categories are performing

category_performance = (
    revenue_data
    .groupby("Category")
    .agg(
        Revenue=("Revenue", "sum"),
        Units=("Quantity", "sum"),
        Orders=("OrderID", "nunique")
    )
    .reset_index()
)

category_performance = (
    category_performance
    .sort_values(
        "Revenue",
        ascending=False
    )
)

display(category_performance)

In [0]:
category_performance["RevenueShare"] = (
    category_performance["Revenue"]
    / category_performance["Revenue"].sum()
    * 100
)

category_performance.sort_values(
    "Units",
    ascending=False
)

In [0]:
category_performance["RevenuePerUnit"] = (
    category_performance["Revenue"]
    / category_performance["Units"]
)
display(category_performance)

In [0]:
## Revenue By Category Chart

plt.figure(figsize=(10, 5))

plt.bar(
    category_performance["Category"],
    category_performance["Revenue"]
)

plt.title("Revenue by Category")
plt.xlabel("Category")
plt.ylabel("Revenue (R)")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


In [0]:
# Top 5 Performing products Revenue Chart
top_5 = (
    product_performance
    .sort_values("Revenue", ascending=False)
    .head(5)
)

plt.figure(figsize=(10, 5))

plt.bar(
    top_5["ProductName"],
    top_5["Revenue"]
)

plt.title("Top 5 Products by Revenue")
plt.xlabel("Product")
plt.ylabel("Revenue (R)")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [0]:
# Top 5 Performing Products by units

top_5_units = (
    product_performance
    .sort_values("Units", ascending=False)
    .head(5)
)

plt.figure(figsize=(10, 5))

plt.bar(
    top_5_units["ProductName"],
    top_5_units["Units"]
)

plt.title("Top 5 Products by Units Sold")
plt.xlabel("Product")
plt.ylabel("Units Sold")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

6. Cancellations , Returns & Payments

In [0]:
# Order Status

orders_clean["Status"].value_counts()

In [0]:
# Calculating the  proportion of the order status distribution

status_summary = (
    orders_clean["Status"]
    .value_counts()
    .reset_index()
)

status_summary.columns = [
    "Status",
    "Orders"
]

status_summary["Percentage"] = (
    status_summary["Orders"]
    / len(orders_clean)
    * 100
)
status_summary

In [0]:
#Combine Cancelled & Returned orders proportion

problem_orders = orders_clean[
    orders_clean["Status"].isin(
        ["Cancelled", "Returned"]
    )
]

problem_order_rate = (
    len(problem_orders)
    / len(orders_clean)
    * 100
)

print(problem_order_rate)

#8.0% of orders were either cancelled or returned.

ORDER STATUS CHART

In [0]:
plt.figure(figsize=(8, 5))

plt.bar(
    status_summary["Status"],
    status_summary["Orders"]
)

plt.title("Order Status")
plt.xlabel("Order Status")
plt.ylabel("Number of Orders")

plt.tight_layout()
plt.show()

In [0]:


#Inspecting payment Performance
payments["PaymentStatus"].value_counts()

In [0]:
failed_payments = (
    payments["PaymentStatus"] == "Failed"
).sum()

total_payments = len(payments)

failure_rate = (
    failed_payments
    / total_payments
    * 100
)

print(failure_rate)

#3.85% of payment attempts failed.

In [0]:
# Payments status proportion
payment_summary = (
    payments["PaymentStatus"]
    .value_counts()
    .reset_index()
)

payment_summary.columns = [
    "PaymentStatus",
    "Payments"
]

payment_summary["Percentage"] = (
    payment_summary["Payments"]
    / len(payments)
    * 100
)

display(payment_summary)

In [0]:
#Payment Methoda analysis

payment_method_analysis = (
    shop
    .dropna(subset=["PaymentMethod"])
    .groupby("PaymentMethod")
    .agg(
        PaymentAttempts=("OrderID", "nunique"),
        Failed=("PaymentStatus",
                lambda x: (x == "Failed").sum()),
        Paid=("PaymentStatus",
             lambda x: (x == "Paid").sum()),
        Refunded=("PaymentStatus",
                  lambda x: (x == "Refunded").sum())
    )
    .reset_index()
)

display(payment_method_analysis)

In [0]:
# Calculating failure rate by method

payment_method_analysis["FailureRate"] = (
    payment_method_analysis["Failed"]
    / payment_method_analysis["PaymentAttempts"]
    * 100
)
display(payment_method_analysis)

Payment Failure Chart


In [0]:
plt.figure(figsize=(9, 5))

plt.bar(
    payment_method_analysis["PaymentMethod"],
    payment_method_analysis["FailureRate"]
)

plt.title("Payment Failure Rate by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Failure Rate (%)")

plt.tight_layout()
plt.show()

Payment Status Chart

In [0]:
plt.figure(figsize=(8, 5))

plt.bar(
    payment_summary["PaymentStatus"],
    payment_summary["Payments"]
)

plt.title("Payment Status")
plt.xlabel("Payment Status")
plt.ylabel("Number of Payments")

plt.tight_layout()
plt.show()

 6. Customers, Cities & Customer Segments

In [0]:
#Revenue by Customer Segment
segment_performance = (
    revenue_data
    .groupby("CustomerSegment")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("OrderID", "nunique"),
        Units=("Quantity", "sum"),
        Customers=("CustomerID", "nunique")
    )
    .reset_index()
)
display(segment_performance)

In [0]:
# Sort the Customer's Segements by Revenue

segment_performance = (
    segment_performance
    .sort_values("Revenue", ascending=False)
)
display(segment_performance)

In [0]:
# Calculating Average Order Valu (revenue)e by Segment

segment_performance["AOV"] = (
    segment_performance["Revenue"]
    / segment_performance["Orders"]
)
display(segment_performance)



In [0]:
# Calculating the revenue share porportion

segment_performance["RevenueShare"] = (
    segment_performance["Revenue"]
    / revenue_data["Revenue"].sum()
    * 100
)
display(segment_performance)

Customer Segment Chart





In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.bar(
    segment_performance["CustomerSegment"],
    segment_performance["Revenue"]
)

plt.title("Revenue by Customer Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Revenue (R)")

plt.tight_layout()
plt.show()

In [0]:
###City Analysis..determining which city generate most revenue

city_data = revenue_data.dropna(
    subset=["City"]
).copy()

In [0]:
# Grouping by City

city_performance = (
    city_data
    .groupby("City")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("OrderID", "nunique"),
        Units=("Quantity", "sum"),
        Customers=("CustomerID", "nunique")
    )
    .reset_index()
)

city_performance = (
    city_performance
    .sort_values("Revenue", ascending=False)
)

display(city_performance)

In [0]:
city_performance["Average Order Value"] = (
    city_performance["Revenue"]
    / city_performance["Orders"]
)

display(city_performance)

In [0]:
# Missing City Data with actvive orders and Revenue

missing_city = revenue_data[
    revenue_data["City"].isna()
]

print(missing_city["OrderID"].nunique())
print(missing_city["Revenue"].sum())


Top 5 Performing City Chart

In [0]:
top_cities = (
    city_performance
    .head(5)
)

plt.figure(figsize=(9, 5))

plt.bar(
    top_cities["City"],
    top_cities["Revenue"]
)

plt.title("Top 5 Cities by Revenue")
plt.xlabel("City")
plt.ylabel("Revenue (R)")

plt.tight_layout()
plt.show()

8. Discount Analysis

In [0]:
## A view of our revenue data
revenue_data.head()

In [0]:
# Inspecting the statistical structure of our data
revenue_data["Discount"].describe()

In [0]:
revenue_data["Discount"].value_counts().sort_index()

In [0]:

import numpy as np



In [0]:
# Creating Discount Bands / Segments

bins = [-0.001, 0, 0.05, 0.10, 0.20, 0.30]

labels = [
    "0%",
    "1–5%",
    "6–10%",
    "11–20%",
    "21–30%"
]


revenue_data["Discount_segment"] = np.select(
    [
        revenue_data["Discount"] == 0,
        (revenue_data["Discount"] > 1) & (revenue_data["Discount"] <= 5),
        (revenue_data["Discount"] > 6) & (revenue_data["Discount"] <= 10),
        (revenue_data["Discount"] > 11) & (revenue_data["Discount"] <= 20),
        (revenue_data["Discount"] > 21) & (revenue_data["Discount"] <= 30)
    ],
    
    [
        "No discount",
        "Small discount",
        "Moderate discount",
        "Large discount",
        "Very large discount"
    ],
    default="Outside range"
)


revenue_data["DiscountBand"] = pd.cut(
    revenue_data["Discount"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [0]:
revenue_data[
    ["Discount", "DiscountBand"]
].head(20)

In [0]:
#how many orders are in dsicounted bands

revenue_data["DiscountBand"].value_counts(
    sort=False
)

In [0]:


discount_analysis = (
    revenue_data
    .groupby("DiscountBand", observed=False)
    .agg(
        Orders=("OrderID", "nunique"),
        AvgDiscount=("Discount", "mean"),
        AvgUnits=("Quantity", "mean"),
        TotalUnits=("Quantity", "sum"),
        Revenue=("Revenue", "sum"),
        AvgRevenue=("Revenue", "mean")
    )
    .reset_index()
)
display(discount_analysis)

In [0]:
# Correlation between discount & quantity

discount_quantity_corr = (
    revenue_data["Discount"]
    .corr(revenue_data["Quantity"])
)

print(discount_quantity_corr)

In [0]:
# Revenue share of each discount band

discount_analysis["RevenueShare"] = (
    discount_analysis["Revenue"]
    / revenue_data["Revenue"].sum()
    * 100
)
display(discount_analysis)

In [0]:
#Calculating gross sales before discount
revenue_data["GrossSales"] = (
    revenue_data["Quantity"]
    * revenue_data["UnitPrice"]
)

discount_analysis = (
    revenue_data
    .groupby("DiscountBand", observed=False)
    .agg(
        Orders=("OrderID", "nunique"),
        AvgUnits=("Quantity", "mean"),
        GrossSales=("GrossSales", "sum"),
        Revenue=("Revenue", "sum"),
        AvgRevenue=("Revenue", "mean")
    )
    .reset_index()
)



In [0]:
discount_analysis["DiscountValue"] = (
    discount_analysis["GrossSales"]
    - discount_analysis["Revenue"]
)


display(discount_analysis)

In [0]:
#Discount vs Revenue correlation

discount_revenue_corr = (
    revenue_data["Discount"]
    .corr(revenue_data["Revenue"])
)

print(discount_revenue_corr)

discount with order quantity Chart

In [0]:
plt.figure(figsize=(9, 5))

plt.bar(
    discount_analysis["DiscountBand"],
    discount_analysis["AvgUnits"]
)

plt.title("Average Units per Order by Discount Band")
plt.xlabel("Discount Band")
plt.ylabel("Average Units per Order")

plt.tight_layout()
plt.show()

Revenue per order Chart

In [0]:
plt.figure(figsize=(9, 5))

plt.barh(
    discount_analysis["DiscountBand"],
    discount_analysis["AvgRevenue"]
)

plt.title("Average Revenue per Order by Discount Band")
plt.xlabel("Discount Band")
plt.ylabel("Average Revenue per Order (R)")

plt.tight_layout()
plt.show()

Discount Vs Revenue Scatter Chart

In [0]:
plt.figure(figsize=(8, 5))

plt.scatter(
    revenue_data["Discount"],
    revenue_data["Revenue"],
    alpha=0.3
)

plt.title("Discount vs Order Revenue")
plt.xlabel("Discount")
plt.ylabel("Revenue (R)")

plt.tight_layout()
plt.show()